# Constraint Comparison via Fisher Information

This notebook fits a binary system using **multiple invariant-point constraint combinations** (via `fit_parameters(n_opts=N)`) and computes the **Fisher Information Matrix (FIM)** at each fitted parameter set. This reveals which constraint combination leads to parameters that are best-supported by the experimental liquidus data.

## Workflow
1. Load a binary system with a full experimental liquidus (digitized from MPDS).
2. Run `fit_parameters(n_opts=N)` — each attempt uses a different combination of invariant-point constraints (eutectics, peritectics, congruent-melting points) as boundary conditions during fitting.
3. For each fitting result, compute the FIM at the fitted parameter values using the full experimental liquidus as data.
4. Compare results: per-parameter uncertainty, overall information content (det FIM), and eigenvalue spectra.

## Interpretation
- A higher det(FIM) means the fitted parameters are more tightly constrained by the liquidus data — the model is more identifiable at that set of parameter values.
- Smaller per-parameter σ means the data more strongly constrains that individual parameter.
- The eigenvalue spectrum shows whether there is a near-unidentifiable direction in parameter space (small minimum eigenvalue = one combination of parameters is poorly constrained).

> **Note:** For `comb-exp` format, all fits share the same free parameters (L0\_b and L1\_a, indices 1 and 2). The comparison reflects how different invariant-point constraints steer the optimizer to regions of parameter space with different data-sensitivity.

In [ ]:
import os
import copy
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
from IPython.display import display, Markdown

from gliquid.binary import BinaryLiquid, BLPlotter
from gliquid.fisher_information import (
    compute_fim,
    compare_constraint_sets,
    ConstraintComparisonResult,
)

os.environ["NEW_MP_API_KEY"] = "YOUR_API_KEY_HERE"

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

SYSTEM       = "Cu-Mg"      # System with a full MPDS experimental liquidus
N_OPTS       = 5            # Number of constraint combinations to try
MAX_ITER     = 64           # Nelder-Mead iterations per fitting attempt
SIGMA        = 5.0          # Assumed temperature measurement uncertainty (K)
PARAM_FORMAT = 'comb-exp'   # Parameter format

print(f"System: {SYSTEM}  |  n_opts = {N_OPTS}  |  max_iter = {MAX_ITER}")
print(f"σ = {SIGMA} K  |  format = {PARAM_FORMAT}")

In [ ]:
# ============================================================
# STEP 1: Load system
# ============================================================

bl = BinaryLiquid.from_cache(SYSTEM, param_format=PARAM_FORMAT)
print(f"Loaded {SYSTEM}: {len(bl.digitized_liq)} digitized liquidus points")

# Show DFT convex hull prediction as a reference
BLPlotter(bl).show('pred')

In [ ]:
# ============================================================
# STEP 2: Multi-constraint fitting
# ============================================================

print(f"Running fit_parameters(n_opts={N_OPTS}, max_iter={MAX_ITER}) for {SYSTEM}...")
print("(This may take several minutes)")

fit_results = bl.fit_parameters(n_opts=N_OPTS, max_iter=MAX_ITER, verbose=False)

print(f"\n{len(fit_results)} fitting attempt(s) completed.")
print()
header = f"{'#':<4} {'Constraint combination':<36} {'MAE (K)':<10} {'L0_a':<12} {'L0_b':<10} {'L1_a':<12} {'L1_b':<10}"
print(header)
print('-' * len(header))
for i, r in enumerate(fit_results):
    print(f"{i+1:<4} {r['constrs']:<36} {r['mae']:<10.1f} "
          f"{r['L0_a']:<12.1f} {r['L0_b']:<10.4f} {r['L1_a']:<12.1f} {r['L1_b']:<10.4f}")

In [ ]:
# ============================================================
# STEP 3: Reconstruct BinaryLiquid for each fitting result
# ============================================================
# Each bl_copy gets the fitted parameters from one attempt.
# The FIM will be computed at those parameter values using
# the same experimental liquidus compositions for all copies.

x_liq = np.array([pt[0] for pt in bl.digitized_liq if 0.0 < pt[0] < 1.0])
print(f"Using {len(x_liq)} experimental liquidus compositions for FIM evaluation.")

bl_fits = []
labels  = []
for i, r in enumerate(fit_results):
    bc = copy.deepcopy(bl)
    bc.update_params([r['L0_a'], r['L0_b'], r['L1_a'], r['L1_b']])
    bl_fits.append(bc)
    labels.append(f"#{i+1} {r['constrs'][:26]}  MAE={r['mae']:.1f}K")

print(f"Prepared {len(bl_fits)} BinaryLiquid copies.")

In [ ]:
# ============================================================
# STEP 4: Compute FIM for each fitting result
# ============================================================

print("Computing FIM for each fitting result (may take ~30–60 s)...")
comparison = compare_constraint_sets(bl_fits, labels, x_compositions=x_liq, sigma=SIGMA)
print("Done.")

# Identify free parameter names (non-nan across all fits)
free_param_names = [
    name for name, vals in comparison.param_variance_table.items()
    if not np.all(np.isnan(vals))
]
print(f"Free parameters across all fits: {free_param_names}")

In [ ]:
# ============================================================
# STEP 5: Combined 2×2 visualization
# ============================================================

n_fits      = len(labels)
short_labels = [l[:30] for l in labels]
colors      = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#AA3377',
                '#66CCEE', '#BBBBBB', '#332288']

fig = sp.make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Per-parameter 1σ uncertainty  (smaller = better constrained)",
        "Overall information: det(FIM)  (larger = more identifiable)",
        "FIM eigenvalue spectra  (log scale)",
        "Fitted parameter values with ±1σ error bars",
    ],
    row_heights=[0.5, 0.5],
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

param_colors = ['#4477AA', '#EE6677', '#228833', '#CCBB44']

# --- Top-left: grouped bar chart of per-parameter σ ---
for pi, pname in enumerate(free_param_names):
    stds = [np.sqrt(v) if not np.isnan(v) else 0.0
            for v in comparison.param_variance_table[pname]]
    fig.add_trace(
        go.Bar(x=short_labels, y=stds, name=pname,
               marker_color=param_colors[pi % len(param_colors)]),
        row=1, col=1
    )

# --- Top-right: det(FIM) bar chart ---
fig.add_trace(
    go.Bar(x=short_labels, y=comparison.det_fim_values,
           marker_color=[colors[i % len(colors)] for i in range(n_fits)],
           name='det(FIM)', showlegend=False),
    row=1, col=2
)

# --- Bottom-left: eigenvalue spectra ---
for i, (label, evs) in enumerate(zip(short_labels, comparison.eigenvalue_table)):
    valid_evs = evs[~np.isnan(evs)]
    if len(valid_evs) == 0:
        continue
    fig.add_trace(
        go.Scatter(
            x=list(range(1, len(valid_evs) + 1)),
            y=valid_evs,
            name=label,
            mode='markers+lines',
            marker=dict(color=colors[i % len(colors)], size=10),
            line=dict(color=colors[i % len(colors)]),
            showlegend=False,
        ),
        row=2, col=1
    )

# --- Bottom-right: fitted parameter values with ±1σ error bars ---
# Show one subplot per free parameter, overlaid on same axes
for pi, pname in enumerate(free_param_names):
    param_idx = ['L0_a', 'L0_b', 'L1_a', 'L1_b'].index(pname)
    vals = [r.fim_results[i].fim_inv is not None and  # guard
            bl_fits[i]._params[param_idx]
            for i in range(n_fits)]
    vals = [bl_fits[i]._params[param_idx] for i in range(n_fits)]
    errs = [np.sqrt(comparison.param_variance_table[pname][i])
            if not np.isnan(comparison.param_variance_table[pname][i]) else 0.0
            for i in range(n_fits)]
    fig.add_trace(
        go.Scatter(
            x=short_labels,
            y=vals,
            error_y=dict(type='data', array=errs, visible=True),
            name=pname,
            mode='markers',
            marker=dict(color=param_colors[pi % len(param_colors)], size=10),
            showlegend=False,
        ),
        row=2, col=2
    )

# Layout
fig.update_yaxes(title_text="1σ uncertainty", row=1, col=1)
fig.update_yaxes(title_text="det(FIM)", row=1, col=2)
fig.update_xaxes(title_text="Eigenvalue index", row=2, col=1)
fig.update_yaxes(title_text="Eigenvalue", type='log', row=2, col=1)
fig.update_yaxes(title_text="Parameter value ± 1σ", row=2, col=2)
fig.update_xaxes(tickangle=-30)
fig.update_layout(
    title=f"{SYSTEM}: Fisher Information comparison across {n_fits} constraint combinations",
    barmode='group',
    template='simple_white',
    height=850,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [ ]:
# ============================================================
# STEP 6: Summary table
# ============================================================

rows = []
header_cols = ['#', 'Constraint combination', 'MAE (K)', 'det(FIM)', 'Cond #']
for pname in free_param_names:
    header_cols.append(f'σ({pname})')

rows.append('| ' + ' | '.join(header_cols) + ' |')
rows.append('| ' + ' | '.join(['---'] * len(header_cols)) + ' |')

for i, (label, r) in enumerate(zip(labels, fit_results)):
    det_val  = comparison.det_fim_values[i]
    cond_val = comparison.condition_numbers[i]
    row_vals = [
        str(i + 1),
        r['constrs'][:35],
        f"{r['mae']:.1f}",
        f"{det_val:.3e}",
        f"{cond_val:.1f}",
    ]
    for pname in free_param_names:
        v = comparison.param_variance_table[pname][i]
        row_vals.append(f"{np.sqrt(v):.4f}" if not np.isnan(v) else '—')
    rows.append('| ' + ' | '.join(row_vals) + ' |')

display(Markdown('\n'.join(rows)))

# Highlight best fit
best_det_idx = int(np.argmax(comparison.det_fim_values))
best_mae_idx = int(np.argmin([r['mae'] for r in fit_results]))
print(f"\nHighest det(FIM) → fit #{best_det_idx + 1}: {labels[best_det_idx]}")
print(f"Lowest MAE       → fit #{best_mae_idx + 1}: {labels[best_mae_idx]}")
if best_det_idx == best_mae_idx:
    print("✓ Same fit is best by both criteria.")
else:
    print("! Different fits win on MAE vs. identifiability — worth inspecting both.")